# Training Liveness Detection (Anti-Spoofing)

Training and Evaluation pipeline for the MobileNetV2 anti-spoofing liveness detector.

In [ ]:
import torch
from src.liveness.liveness_detector import LivenessDetector

model = LivenessDetector()
print('Liveness model initialized:', model)

### Step 1: Configure Data Loaders
Load the processed dataset, split it into training (80%) and validation (20%) sets, and define PyTorch DataLoaders.

In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Dataset path (defaults to Colab local directory /content/liveness_dataset)
dataset_path = "/content/liveness_dataset"

# Define image pre-processing and data augmentations
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
if os.path.exists(dataset_path):
    full_dataset = datasets.ImageFolder(root=dataset_path, transform=data_transforms)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    print(f"Loaded {len(full_dataset)} images. Train: {len(train_dataset)}, Val: {len(val_dataset)}")
else:
    print(f"Warning: Dataset path '{dataset_path}' not found. Please verify it has been processed.")

### Step 2: Run the Training Loop
Execute the training function to train the liveness model using binary cross-entropy loss.

In [ ]:
import logging
from src.liveness.train_liveness import train_liveness_model

logging.basicConfig(level=logging.INFO)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if 'train_loader' in globals() and 'val_loader' in globals():
    trained_model = train_liveness_model(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=10,
        device=device
    )
else:
    print("Error: Data loaders not initialized. Run Step 1 first.")

### Step 3: Export Weights File
Save the trained model weights directly so they can be loaded by the main application.

In [ ]:
# Copy the saved weights to a persistent location if needed
if os.path.exists("models/liveness/best_liveness.pth"):
    print("Model trained successfully! Saved to: models/liveness/best_liveness.pth")
else:
    print("Model weights file not found.")